# Brainrot Translator Training Notebook

This notebook fine-tunes `google/flan-t5-small` using one final instruction-format dataset.

Expected dataset columns:

- `input_text`
- `target_text`
- `task_type`
- `source`
- `quality_label`
- `reason`

Set only `FINAL_DATASET_PATH` below when you want to change datasets.


In [ ]:
# Install the training dependencies.
# Pin transformers so the notebook matches the runtime behavior fixed below.
!pip install transformers==5.8.1 datasets accelerate sentencepiece evaluate


In [ ]:
# Import the libraries and define the single dataset path used by the notebook.
from pathlib import Path
from difflib import SequenceMatcher
import math
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

FINAL_DATASET_PATH = "/content/drive/MyDrive/brainrot_translator/data/processed/training_dataset_final_combined.csv"
OUTPUT_DIR = Path("/content/drive/MyDrive/brainrot_translator/models/brainrot-translator-v1")
PROJECT_DATASET_PATH = Path(FINAL_DATASET_PATH)
PROJECT_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/flan-t5-small"
TRANSLATION_PROMPT_PREFIX = "Convert brainrot English to normal English: "
DEFINITION_PROMPT_PREFIX = "Define this brainrot term in normal English: "
EPOCHS = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.06
GPU_BATCH_SIZE = 4
CPU_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS_GPU = 4
GRADIENT_ACCUMULATION_STEPS_CPU = 8
TERM_DEFINITION_OVERSAMPLE = 3
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
MAX_NEW_TOKENS = 48
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 3
REQUIRE_GPU = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime not detected. In Colab go to Runtime > Change runtime type > T4 GPU, then rerun the notebook."
    )

print(f"Transformers version: {transformers.__version__}")
print(f"Final dataset path: {PROJECT_DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Using GPU: {torch.cuda.is_available()}")


## Optional Data Upload

Use this cell if you want to manually upload the final cleaned CSV into the current runtime.
The notebook saves the uploaded file into `FINAL_DATASET_PATH` so the rest of the notebook can use one stable location.


In [ ]:
# Optional: upload the final cleaned dataset from your computer.
from google.colab import files

uploaded = files.upload()
if uploaded:
    uploaded_name = next(iter(uploaded))
    PROJECT_DATASET_PATH.write_bytes(uploaded[uploaded_name])
    print(f"Saved {uploaded_name} to {PROJECT_DATASET_PATH}")
else:
    print("No file uploaded in this step. Skip this cell if your dataset is already available.")


## Optional Google Drive Mount

Use this cell if your final dataset is stored in Google Drive.
After mounting, copy that file into `FINAL_DATASET_PATH` or change the single path variable above.


In [ ]:
# Optional: mount Google Drive.
from google.colab import drive

drive.mount("/content/drive")
print("If needed, copy your final dataset into FINAL_DATASET_PATH before continuing.")


In [ ]:
# Load the final instruction-format dataset.
if not PROJECT_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {PROJECT_DATASET_PATH}. Create it with scripts/prepare_training_dataset.py or upload it to this path."
    )

df = pd.read_csv(PROJECT_DATASET_PATH)
required_columns = {"input_text", "target_text", "task_type", "source", "quality_label", "reason"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

df = df[["input_text", "target_text", "task_type", "source", "quality_label", "reason"]].copy()
df["input_text"] = df["input_text"].astype(str).str.strip()
df["target_text"] = df["target_text"].astype(str).str.strip()
df["task_type"] = df["task_type"].astype(str).str.strip()
df = df[(df["input_text"] != "") & (df["target_text"] != "")].reset_index(drop=True)
df = df[df["task_type"].isin(["sentence_translation", "term_definition"])].reset_index(drop=True)

display(df.head())
print(f"Total cleaned rows available for training: {len(df)}")
print("Task counts:")
print(df["task_type"].value_counts())
print("Source counts:")
print(df["source"].value_counts().head(10))


In [ ]:
# Split the dataset into training and validation sets while keeping both task types in the split.
def stratified_task_split(dataframe, test_size=0.1, seed=42):
    train_parts = []
    validation_parts = []

    for task_type, group in dataframe.groupby("task_type", dropna=False):
        shuffled = group.sample(frac=1, random_state=seed).reset_index(drop=True)
        if len(shuffled) == 1:
            train_parts.append(shuffled)
            continue

        validation_rows = max(1, int(round(len(shuffled) * test_size)))
        validation_rows = min(validation_rows, len(shuffled) - 1)
        validation_parts.append(shuffled.iloc[:validation_rows])
        train_parts.append(shuffled.iloc[validation_rows:])

    train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    if validation_parts:
        validation_df = pd.concat(validation_parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    else:
        validation_df = dataframe.sample(frac=0, random_state=seed).copy()

    return train_df, validation_df


def oversample_training_tasks(dataframe, task_name, multiplier, seed=42):
    if multiplier <= 1:
        return dataframe.copy()

    target_rows = dataframe[dataframe["task_type"] == task_name].copy()
    other_rows = dataframe[dataframe["task_type"] != task_name].copy()
    if target_rows.empty:
        return dataframe.copy()

    boosted_parts = [target_rows]
    for repeat_index in range(multiplier - 1):
        boosted_parts.append(target_rows.sample(frac=1, replace=True, random_state=seed + repeat_index).reset_index(drop=True))

    boosted_target_rows = pd.concat(boosted_parts, ignore_index=True)
    return pd.concat([other_rows, boosted_target_rows], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


train_df, validation_df = stratified_task_split(df, test_size=0.1, seed=42)
train_df = oversample_training_tasks(train_df, "term_definition", TERM_DEFINITION_OVERSAMPLE, seed=42)
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_df, preserve_index=False)

print(f"Training rows after oversampling: {len(train_df)}")
print(train_df["task_type"].value_counts())
print(f"Validation rows: {len(validation_df)}")
print(validation_df["task_type"].value_counts())


In [ ]:
# Load the FLAN-T5 tokenizer and a clean base pretrained model.
# The model is always loaded from the original checkpoint, not from a local fine-tuned folder.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

try:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=False,
        attn_implementation="eager",
    )
    print("Loaded model with eager attention.")
except TypeError:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=False,
    )
    print("Loaded model without eager attention because this runtime does not support it.")

model.config.use_cache = False
model.config.decoder_start_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.float().to(device)

print(f"Loaded model: {MODEL_NAME}")
print(f"Model device: {model.device}")
print(f"Model dtype: {next(model.parameters()).dtype}")


In [ ]:
# Tokenize the instruction-format rows directly from the final dataset.
def preprocess_examples(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_examples,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_validation = validation_dataset.map(
    preprocess_examples,
    batched=True,
    remove_columns=validation_dataset.column_names,
)

print(tokenized_train)


In [ ]:
# Run a small forward-pass diagnostic before training.
# If CUDA produces NaN or Inf, the notebook automatically falls back to CPU.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


def test_forward_pass(model, dataset, data_collator, batch_size=4):
    batch = data_collator([dataset[i] for i in range(min(batch_size, len(dataset)))])
    batch = {k: v.to(model.device) for k, v in batch.items()}
    model.eval()
    with torch.no_grad():
        outputs = model(**batch)
    loss_is_nan = torch.isnan(outputs.loss).item()
    logits_has_nan = torch.isnan(outputs.logits).any().item()
    logits_has_inf = torch.isinf(outputs.logits).any().item()
    print("Device:", model.device)
    print("Loss:", outputs.loss)
    print("Loss is NaN:", loss_is_nan)
    print("Logits has NaN:", logits_has_nan)
    print("Logits has Inf:", logits_has_inf)
    return not (loss_is_nan or logits_has_nan or logits_has_inf)


forward_ok = test_forward_pass(model, tokenized_train, data_collator)

if not forward_ok and torch.cuda.is_available():
    print("CUDA forward pass is unstable. Falling back to CPU.")
    model = model.cpu().float()
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    forward_ok = test_forward_pass(model, tokenized_train, data_collator)

if not forward_ok:
    raise RuntimeError("Forward pass still produces NaN/Inf. Stop training and inspect model/data.")


def run_instruction(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(model.device)
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=4,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            early_stopping=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def translate_brainrot(text):
    return run_instruction(TRANSLATION_PROMPT_PREFIX + text)


def define_brainrot_term(term):
    return run_instruction(DEFINITION_PROMPT_PREFIX + term)


print("Pre-training translation check:")
print(translate_brainrot("bro cooked so hard fr"))
print("Pre-training definition check:")
print(define_brainrot_term("delulu"))


In [ ]:
# Configure the trainer with stable settings for this runtime.
use_cpu_training = model.device.type == "cpu"
gradient_accumulation_steps = (
    GRADIENT_ACCUMULATION_STEPS_CPU if use_cpu_training else GRADIENT_ACCUMULATION_STEPS_GPU
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=CPU_BATCH_SIZE if use_cpu_training else GPU_BATCH_SIZE,
    per_device_eval_batch_size=CPU_BATCH_SIZE if use_cpu_training else GPU_BATCH_SIZE,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=False,
    bf16=False,
    tf32=False,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    use_cpu=use_cpu_training,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(f"Training on device: {model.device}")
print(
    "Train/Eval batch size:",
    training_args.per_device_train_batch_size,
    training_args.per_device_eval_batch_size,
)
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Epochs: {EPOCHS}")


In [ ]:
# Train the model, make sure losses stay finite, then save and zip it.
train_result = trainer.train()
model = trainer.model

post_training_ok = test_forward_pass(model, tokenized_train, data_collator)
if not post_training_ok:
    raise RuntimeError("Model outputs became NaN/Inf after training. Model was not saved.")

eval_metrics = trainer.evaluate()
print("Evaluation metrics:", eval_metrics)

train_loss = train_result.training_loss
eval_loss = eval_metrics.get("eval_loss")

if train_loss is not None:
    print(f"Training loss: {train_loss:.4f}")
if eval_loss is not None:
    print(f"Validation loss: {eval_loss:.4f}")

if (train_loss is not None and not math.isfinite(train_loss)) or (
    eval_loss is not None and not math.isfinite(eval_loss)
):
    raise RuntimeError("Training produced non-finite loss values. Model was not saved.")

print("Post-training translation check:")
print(translate_brainrot("bro cooked so hard fr"))
print("Post-training definition check:")
print(define_brainrot_term("delulu"))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=str(OUTPUT_DIR))
print(f"Saved model folder: {OUTPUT_DIR.resolve()}")
print(f"Zipped model file: {zip_path}")


In [ ]:
# Plot training curves and inspect validation quality for both task types.
def normalize_text_for_eval(text):
    return " ".join(str(text).strip().lower().split())


history_df = pd.DataFrame(trainer.state.log_history)
history_df = history_df.sort_values("step").reset_index(drop=True)
train_curve = history_df.dropna(subset=["loss"])[["step", "loss"]].copy()
eval_curve = history_df.dropna(subset=["eval_loss"])[["step", "eval_loss"]].copy()

prediction_output = trainer.predict(tokenized_validation)
decoded_predictions = tokenizer.batch_decode(prediction_output.predictions, skip_special_tokens=True)

validation_results = validation_df[["input_text", "target_text", "task_type", "source"]].copy().reset_index(drop=True)
validation_results["prediction"] = [prediction.strip() for prediction in decoded_predictions]
validation_results["normalized_prediction"] = validation_results["prediction"].map(normalize_text_for_eval)
validation_results["normalized_target"] = validation_results["target_text"].map(normalize_text_for_eval)
validation_results["exact_match"] = (
    validation_results["normalized_prediction"] == validation_results["normalized_target"]
).astype(float)
validation_results["similarity"] = [
    SequenceMatcher(None, prediction, target).ratio()
    for prediction, target in zip(
        validation_results["normalized_prediction"],
        validation_results["normalized_target"],
    )
]

task_summary = validation_results.groupby("task_type", as_index=False).agg(
    count=("task_type", "size"),
    exact_match_rate=("exact_match", "mean"),
    mean_similarity=("similarity", "mean"),
)

overall_similarity = float(task_summary["mean_similarity"].mean()) if not task_summary.empty else 0.0
if not eval_curve.empty:
    first_eval_loss = float(eval_curve["eval_loss"].iloc[0])
    last_eval_loss = float(eval_curve["eval_loss"].iloc[-1])
    if last_eval_loss <= first_eval_loss and overall_similarity >= 0.55:
        training_status = "Good early sign: validation loss went down and validation similarity is usable."
    elif last_eval_loss <= first_eval_loss:
        training_status = "Loss improved, but inspect the examples because text quality is still mixed."
    else:
        training_status = "Warning: validation loss increased, so check for overfitting or noisy data."
else:
    training_status = "No evaluation checkpoints were logged, so rely on the sample outputs below."

print(training_status)
display(task_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if not train_curve.empty:
    axes[0].plot(train_curve["step"], train_curve["loss"], marker="o", label="train_loss")
if not eval_curve.empty:
    axes[0].plot(eval_curve["step"], eval_curve["eval_loss"], marker="s", label="eval_loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].bar(task_summary["task_type"], task_summary["mean_similarity"], color=["#2a9d8f", "#457b9d"][: len(task_summary)])
axes[1].set_title("Validation Similarity by Task")
axes[1].set_ylabel("Mean similarity")
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", alpha=0.3)

axes[2].bar(task_summary["task_type"], task_summary["exact_match_rate"], color=["#f4a261", "#e76f51"][: len(task_summary)])
axes[2].set_title("Validation Exact Match by Task")
axes[2].set_ylabel("Exact match rate")
axes[2].set_ylim(0, 1)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Best validation examples")
display(
    validation_results.sort_values("similarity", ascending=False)[
        ["task_type", "input_text", "prediction", "target_text", "similarity"]
    ].head(5)
)

print("Worst validation examples")
display(
    validation_results.sort_values("similarity", ascending=True)[
        ["task_type", "input_text", "prediction", "target_text", "similarity"]
    ].head(5)
)


In [ ]:
# Optional: download the zipped model directly from Colab.
from google.colab import files

files.download(f"{OUTPUT_DIR}.zip")


In [ ]:
# Test the trained model with both task types.
model = trainer.model
print(f"Final model device: {model.device}")

translation_examples = [
    "bro is cooked fr no cap",
    "this code is tweaking",
    "she ate and left no crumbs",
    "that assignment lowkey cooked me",
]

definition_examples = [
    "delulu",
    "fanum tax",
    "GOAT",
    "Roman Empire",
]

print("Translation examples")
for example in translation_examples:
    print(f"Input: {example}")
    print(f"Output: {translate_brainrot(example)}")
    print("-" * 60)

print("Definition examples")
for example in definition_examples:
    print(f"Input: {example}")
    print(f"Output: {define_brainrot_term(example)}")
    print("-" * 60)
